# Clase 050 — Panorama del ML: tipos, batch vs online, instance vs model-based

Ubicamos cada algoritmo en la grilla `(supervisión × batch/online × instance/model)`.
Comparamos KNN (instance-based) contra LogisticRegression (model-based) y mostramos el
patrón canónico de *online learning* con `partial_fit`.

Requiere: `numpy`, `pandas`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_regression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error

np.random.seed(42)
print('setup ok')

## 1. Supervisado: KNN (instance-based) vs LogReg (model-based)

Ambos resuelven el mismo problema supervisado (clasificar Iris) pero generalizan distinto:
KNN compara por similitud contra los vistos; LogReg aprende parámetros `theta`.

In [ ]:
iris = load_iris()
Xtr, Xte, ytr, yte = train_test_split(iris.data, iris.target, test_size=0.3,
                                      stratify=iris.target, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5).fit(Xtr, ytr)
logreg = LogisticRegression(max_iter=1000).fit(Xtr, ytr)

acc_knn = accuracy_score(yte, knn.predict(Xte))
acc_lr = accuracy_score(yte, logreg.predict(Xte))
print(f'KNN     accuracy test: {acc_knn:.4f}')
print(f'LogReg  accuracy test: {acc_lr:.4f}')

## 2. Instance-based pesa más: evidencia empírica por tamaño serializado

KNN "memoriza" el train set entero; su modelo serializado crece con N. LogReg guarda solo
coeficientes. Verificamos con `pickle.dumps` que el KNN ocupa más bytes.

In [ ]:
bytes_knn = len(pickle.dumps(knn))
bytes_lr = len(pickle.dumps(logreg))
print(f'KNN    serializado: {bytes_knn:5d} bytes')
print(f'LogReg serializado: {bytes_lr:5d} bytes')
assert bytes_knn > bytes_lr, 'KNN deberia pesar mas: guarda el train set'
print('OK: el instance-based ocupa mas porque ES su dataset')

## 3. Batch vs online: `SGDRegressor.partial_fit` en mini-batches

Online learning actualiza el modelo instancia a instancia (o en mini-batches) vía
`partial_fit`. Es el patrón para *streaming* y *out-of-core*. Ploteamos el MSE de train
a medida que llegan los batches.

In [ ]:
X, y = make_regression(n_samples=5000, n_features=10, noise=15.0, random_state=42)
X = StandardScaler().fit_transform(X)
y = (y - y.mean()) / y.std()

sgd = SGDRegressor(learning_rate='invscaling', eta0=0.01, random_state=42)
batch = 500
mse_hist = []
for start in range(0, len(X), batch):
    xb, yb = X[start:start + batch], y[start:start + batch]
    sgd.partial_fit(xb, yb)
    mse_hist.append(mean_squared_error(y, sgd.predict(X)))
print('batches procesados:', len(mse_hist))
print(f'MSE inicial {mse_hist[0]:.4f} -> MSE final {mse_hist[-1]:.4f}')
assert mse_hist[-1] < mse_hist[0], 'el online learning deberia reducir el error'

## 4. Costo de inferencia: instance-based escala con N

KNN compara cada predicción contra todo el train set. Medimos el tiempo de 1000
predicciones para KNN vs LogReg.

In [ ]:
import time
probe = np.repeat(Xte[:1], 1000, axis=0)

t0 = time.perf_counter(); knn.predict(probe); t_knn = time.perf_counter() - t0
t0 = time.perf_counter(); logreg.predict(probe); t_lr = time.perf_counter() - t0
print(f'KNN    1000 preds: {t_knn*1000:.2f} ms')
print(f'LogReg 1000 preds: {t_lr*1000:.2f} ms')
print('KNN es "lazy": mueve el costo de fit a predict')

## 5. La grilla del capítulo 1: ubicar cada algoritmo

Resumimos la taxonomía en una tabla sobre los tres ejes.

In [ ]:
taxonomia = pd.DataFrame([
    ('KNeighborsClassifier', 'supervisado', 'batch',  'instance'),
    ('LogisticRegression',   'supervisado', 'batch',  'model'),
    ('SGDRegressor',         'supervisado', 'online', 'model'),
    ('KMeans',               'no-superv.',  'batch',  'model'),
    ('RandomForest',         'supervisado', 'batch',  'model'),
], columns=['algoritmo', 'supervision', 'batch/online', 'instance/model'])
print(taxonomia.to_string(index=False))

## 6. Visual: tamaño del modelo y velocidad, instance vs model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['KNN\n(instance)', 'LogReg\n(model)'], [bytes_knn, bytes_lr],
            color=['#c33', '#37a'])
axes[0].set_ylabel('bytes serializados')
axes[0].set_title('Instance-based ocupa mas')

axes[1].plot(np.arange(1, len(mse_hist) + 1), mse_hist, marker='o', color='#3a7')
axes[1].set_xlabel('mini-batch #'); axes[1].set_ylabel('MSE en train')
axes[1].set_title('Online learning: el error baja por batch')
plt.tight_layout(); plt.show()

## Ejercicios

1. Agregá `KNeighborsRegressor` y `Ridge` sobre `make_regression` y compará RMSE en test y
   bytes serializados. Confirmá que el patrón instance vs model se repite en regresión.
2. En el loop de `partial_fit`, barajá los datos con `np.random.permutation` antes de armar
   los batches. ¿Cómo cambia la curva de MSE? ¿Por qué el orden importa en online learning?
3. Cambiá `n_neighbors` de KNN a 1 y a 50. ¿Qué le pasa al accuracy y al tiempo de inferencia?
4. Clasificá en la grilla de 3 ejes: `AlphaGo`, un autoencoder y `SGDClassifier`. Justificá
   cada celda (algunos quedan en casos raros).

## Conclusiones

- El marco `(supervisión × batch/online × instance/model)` ubica cualquier algoritmo y
  anticipa su API, forma de evaluarlo y debilidades.
- KNN es instance-based: su "modelo" es el dataset; pesa más y su predicción escala con N.
- `partial_fit` es el patrón canónico de online learning: ideal para streaming y out-of-core.
- No-supervisado no significa "sin entrenar": KMeans igual ajusta centroides con `.fit(X)`.

## ✅ Soluciones de los ejercicios

Los ejercicios 1, 2 y 5 son conceptuales: los resolvemos codificando la respuesta como datos verificables (diccionarios) y validando su estructura. Los ejercicios 3 y 4 son de código.

**Ej. 1 — Clasificá 6 problemas.** Cada caso etiquetado como supervisado / no-supervisado / semi-supervisado / RL, con su justificación.

In [ ]:

respuestas = {
 "a_fraude_tarjeta":     ("supervisado",     "hay historial etiquetado fraude/no-fraude; se aprende f(X)->y"),
 "b_segmentar_clientes": ("no-supervisado",  "no hay etiqueta; se buscan clusters de comportamiento"),
 "c_traducir_en_es":     ("supervisado",     "pares (frase_en, frase_es) son ejemplos con 'respuesta correcta' (seq2seq)"),
 "d_ajedrez":            ("refuerzo",        "agente + recompensa (ganar/perder); aprende una politica, no una etiqueta fija"),
 "e_caras_duplicadas":   ("semi-supervisado","10k fotos, solo 50 taggeadas: pocas etiquetas + mucho dato crudo"),
 "f_precio_dolar_7d":    ("supervisado",     "regresion de serie temporal: y continuo futuro a partir de X historico"),
}
for k, (tipo, why) in respuestas.items():
    print(f"{k:22s} -> {tipo:17s} | {why}")
assert respuestas["d_ajedrez"][0] == "refuerzo"
assert respuestas["e_caras_duplicadas"][0] == "semi-supervisado"
print("OK: 6 problemas clasificados")

**Ej. 2 — Batch u online.** Decisión + justificación en una línea por caso.

In [ ]:

decision = {
 "a_scoring_trimestral": ("batch",  "se reentrena cada 3 meses; no necesita adaptarse en vivo"),
 "b_recomendador_news":  ("online", "reacciona a clicks en tiempo real; distribucion cambia rapido"),
 "c_imagenes_medicas":   ("batch",  "dataset estable y validado; reproducibilidad y trazabilidad regulatoria"),
 "d_spam_gmail":         ("online", "el spam muta constantemente; conviene adaptacion incremental"),
}
for k, (modo, why) in decision.items():
    print(f"{k:22s} -> {modo:7s} | {why}")
assert decision["b_recomendador_news"][0] == "online"
assert decision["a_scoring_trimestral"][0] == "batch"
print("OK")

**Ej. 3 — KNN vs LogReg en Iris.** Accuracy en test, tamaño serializado y tiempo de inferencia. El instance-based (KNN) pesa más porque *es* su dataset.

In [ ]:

import pickle, time
import numpy as np
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
knn = KNeighborsClassifier(n_neighbors=5).fit(Xtr, ytr)
lr = LogisticRegression(max_iter=1000).fit(Xtr, ytr)

acc_knn = accuracy_score(yte, knn.predict(Xte))
acc_lr = accuracy_score(yte, lr.predict(Xte))
b_knn, b_lr = len(pickle.dumps(knn)), len(pickle.dumps(lr))
probe = np.repeat(Xte[:1], 1000, axis=0)
t0 = time.perf_counter(); knn.predict(probe); t_knn = time.perf_counter() - t0
t0 = time.perf_counter(); lr.predict(probe); t_lr = time.perf_counter() - t0
print(f"KNN    acc={acc_knn:.3f} | {b_knn:5d} bytes | 1000 preds {t_knn*1e3:6.2f} ms")
print(f"LogReg acc={acc_lr:.3f} | {b_lr:5d} bytes | 1000 preds {t_lr*1e3:6.2f} ms")
assert b_knn > b_lr, "el instance-based (KNN) guarda el train set -> pesa mas"
print("KNN es instance-based: el modelo ES el dataset (mas bytes, prediccion escala con N)")

**Ej. 4 — Out-of-core con `SGDRegressor.partial_fit`.** California Housing no está disponible sin internet, así que usamos un dataset sintético equivalente para mostrar el patrón canónico de online learning: entrenar por mini-batches.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error

X, y = make_regression(n_samples=8000, n_features=8, noise=15.0, random_state=42)
X = StandardScaler().fit_transform(X)
y = (y - y.mean()) / y.std()

sgd = SGDRegressor(learning_rate="invscaling", eta0=0.01, random_state=42)
batch = 500
mse_hist = []
for start in range(0, len(X), batch):
    sgd.partial_fit(X[start:start+batch], y[start:start+batch])
    mse_hist.append(mean_squared_error(y, sgd.predict(X)))
print(f"batches={len(mse_hist)} | MSE inicial {mse_hist[0]:.3f} -> final {mse_hist[-1]:.3f}")
assert mse_hist[-1] < mse_hist[0], "el online learning debe reducir el error batch a batch"
plt.plot(range(1, len(mse_hist)+1), mse_hist, "o-")
plt.xlabel("mini-batch #"); plt.ylabel("MSE train"); plt.title("Online learning (partial_fit)"); plt.show()

**Ej. 5 — Mapa mental (grilla 2×2×2).** Ubicamos cada algoritmo en los ejes `(supervisión, batch/online, instance/model)`. Algunos caen en celdas 'raras' — se anota por qué.

In [ ]:

import pandas as pd
grilla = pd.DataFrame([
 ("KNN",              "supervisado",   "batch",  "instance", "el modelo es el dataset"),
 ("Regresion lineal", "supervisado",   "batch",  "model",    "aprende coeficientes theta"),
 ("k-means",          "no-supervisado","batch",  "model",    "guarda centroides, no instancias"),
 ("Random forest",    "supervisado",   "batch",  "model",    "guarda arboles = forma funcional"),
 ("SGDClassifier",    "supervisado",   "online", "model",    "partial_fit -> incremental"),
 ("AlphaGo",          "refuerzo",      "online", "model",    "caso raro: RL + red que se actualiza jugando"),
 ("Autoencoder",      "auto-superv.",  "batch",  "model",    "caso raro: se inventa su propia etiqueta (reconstruir la entrada)"),
], columns=["algoritmo", "supervision", "batch/online", "instance/model", "nota"])
print(grilla.to_string(index=False))
assert (grilla["instance/model"] == "instance").sum() == 1, "solo KNN es instance-based aca"
assert set(grilla["supervision"]) >= {"supervisado", "no-supervisado", "refuerzo"}
print("OK: grilla 2x2x2 poblada")